In [0]:
%sql
CREATE OR REPLACE TABLE medical_insurance.gold.hospital_performance AS
WITH hospital_base AS (
  SELECT 
    h.hospital_id,
    h.hospital_name,
    h.hospital_type,
    h.governorate,
    h.district,
    h.total_beds,
    h.icu_capacity,
    h.manager_name
  FROM medical_insurance.silver.hospital_silver h
),
visit_stats AS (
  SELECT 
    v.hospital_id,
    COUNT(DISTINCT v.visit_id) AS total_visits,
    COUNT(DISTINCT CASE WHEN v.visit_type = 'Outpatient' THEN v.visit_id END) AS outpatient_visits,
    COUNT(DISTINCT CASE WHEN v.visit_type = 'Inpatient' THEN v.visit_id END) AS inpatient_visits,
    COUNT(DISTINCT CASE WHEN v.visit_type = 'Follow-up' THEN v.visit_id END) AS followup_visits,
    COUNT(DISTINCT v.patient_id) AS unique_patients,
    COUNT(DISTINCT v.doctor_id) AS unique_doctors,
    AVG(v.waiting_time) AS avg_waiting_time_minutes,
    SUM(v.total_amount) AS total_revenue,
    AVG(v.total_amount) AS avg_revenue_per_visit,
    MIN(v.visit_date) AS first_visit_date,
    MAX(v.visit_date) AS last_visit_date
  FROM medical_insurance.silver.visit_silver v
  GROUP BY v.hospital_id
),
bed_stats AS (
  SELECT 
    b.hospital_id,
    COUNT(DISTINCT b.bed_id) AS total_beds_tracked,
    COUNT(DISTINCT CASE WHEN b.availability_status = 'Available' THEN b.bed_id END) AS available_beds,
    COUNT(DISTINCT CASE WHEN b.availability_status = 'Occupied' THEN b.bed_id END) AS occupied_beds,
    COUNT(DISTINCT CASE WHEN b.bed_type = 'ICU' THEN b.bed_id END) AS icu_beds,
    COUNT(DISTINCT CASE WHEN b.bed_type = 'Standard' THEN b.bed_id END) AS standard_beds
  FROM medical_insurance.silver.bed_silver b
  GROUP BY b.hospital_id
),
icu_latest AS (
  SELECT 
    hospital_id,
    occupied_beds AS latest_icu_occupied,
    available_beds AS latest_icu_available,
    update_time AS latest_icu_update
  FROM (
    SELECT 
      hospital_id,
      occupied_beds,
      available_beds,
      update_time,
      ROW_NUMBER() OVER (PARTITION BY hospital_id ORDER BY update_time DESC) AS rn
    FROM medical_insurance.silver.icu_status_silver
  )
  WHERE rn = 1
),
department_stats AS (
  SELECT 
    dept.hospital_id,
    COUNT(DISTINCT dept.department_id) AS total_departments,
    COUNT(DISTINCT CASE WHEN dept.department_name = 'Emergency' THEN dept.department_id END) AS has_emergency_dept
  FROM medical_insurance.silver.department_silver dept
  GROUP BY dept.hospital_id
),
feedback_stats AS (
  SELECT 
    f.hospital_id,
    COUNT(DISTINCT f.feedback_id) AS feedback_count,
    AVG(f.rating) AS avg_patient_rating,
    MIN(f.rating) AS min_rating,
    MAX(f.rating) AS max_rating,
    COUNT(DISTINCT CASE WHEN f.rating >= 4 THEN f.feedback_id END) AS positive_feedback_count,
    COUNT(DISTINCT CASE WHEN f.rating <= 2 THEN f.feedback_id END) AS negative_feedback_count
  FROM medical_insurance.silver.patient_feedback_silver f
  GROUP BY f.hospital_id
),
referral_sent AS (
  SELECT 
    from_hospital_id AS hospital_id,
    COUNT(DISTINCT referral_id) AS referrals_sent
  FROM medical_insurance.silver.referral_silver
  GROUP BY from_hospital_id
),
referral_received AS (
  SELECT 
    to_hospital_id AS hospital_id,
    COUNT(DISTINCT referral_id) AS referrals_received
  FROM medical_insurance.silver.referral_silver
  GROUP BY to_hospital_id
)
SELECT 
  hb.hospital_id,
  hb.hospital_name,
  hb.hospital_type,
  hb.governorate,
  hb.district,
  hb.manager_name,
  
  -- Capacity metrics
  hb.total_beds,
  hb.icu_capacity,
  COALESCE(bs.total_beds_tracked, 0) AS beds_in_system,
  COALESCE(bs.available_beds, 0) AS current_available_beds,
  COALESCE(bs.occupied_beds, 0) AS current_occupied_beds,
  CASE 
    WHEN hb.total_beds > 0 THEN ROUND((bs.occupied_beds * 100.0 / hb.total_beds), 2)
    ELSE 0
  END AS bed_occupancy_rate_pct,
  COALESCE(bs.icu_beds, 0) AS icu_beds_tracked,
  COALESCE(bs.standard_beds, 0) AS standard_beds_tracked,
  
  -- ICU metrics
  COALESCE(icu.latest_icu_occupied, 0) AS latest_icu_occupied,
  COALESCE(icu.latest_icu_available, 0) AS latest_icu_available,
  CASE 
    WHEN hb.icu_capacity > 0 THEN ROUND((icu.latest_icu_occupied * 100.0 / hb.icu_capacity), 2)
    ELSE 0
  END AS icu_occupancy_rate_pct,
  icu.latest_icu_update,
  
  -- Department metrics
  COALESCE(dept.total_departments, 0) AS total_departments,
  CASE WHEN dept.has_emergency_dept > 0 THEN 'Yes' ELSE 'No' END AS has_emergency_department,
  
  -- Visit metrics
  COALESCE(vs.total_visits, 0) AS total_visits,
  COALESCE(vs.outpatient_visits, 0) AS outpatient_visits,
  COALESCE(vs.inpatient_visits, 0) AS inpatient_visits,
  COALESCE(vs.followup_visits, 0) AS followup_visits,
  COALESCE(vs.unique_patients, 0) AS unique_patients_served,
  COALESCE(vs.unique_doctors, 0) AS unique_doctors_count,
  ROUND(COALESCE(vs.avg_waiting_time_minutes, 0), 2) AS avg_waiting_time_minutes,
  vs.first_visit_date,
  vs.last_visit_date,
  
  -- Revenue metrics
  ROUND(COALESCE(vs.total_revenue, 0), 2) AS total_revenue,
  ROUND(COALESCE(vs.avg_revenue_per_visit, 0), 2) AS avg_revenue_per_visit,
  CASE 
    WHEN vs.total_visits > 0 THEN ROUND((vs.total_revenue / vs.total_visits), 2)
    ELSE 0
  END AS revenue_per_visit,
  
  -- Patient satisfaction metrics
  COALESCE(fs.feedback_count, 0) AS feedback_count,
  ROUND(COALESCE(fs.avg_patient_rating, 0), 2) AS avg_patient_rating,
  fs.min_rating,
  fs.max_rating,
  COALESCE(fs.positive_feedback_count, 0) AS positive_feedback_count,
  COALESCE(fs.negative_feedback_count, 0) AS negative_feedback_count,
  CASE 
    WHEN fs.feedback_count > 0 THEN ROUND((fs.positive_feedback_count * 100.0 / fs.feedback_count), 2)
    ELSE 0
  END AS positive_feedback_rate_pct,
  
  -- Referral metrics
  COALESCE(rs.referrals_sent, 0) AS referrals_sent,
  COALESCE(rr.referrals_received, 0) AS referrals_received,
  
  -- Performance indicators
  CASE 
    WHEN vs.total_visits > 10000 THEN 'High Volume'
    WHEN vs.total_visits BETWEEN 5000 AND 10000 THEN 'Medium Volume'
    WHEN vs.total_visits < 5000 THEN 'Low Volume'
    ELSE 'No Activity'
  END AS volume_category,
  
  CASE 
    WHEN bs.occupied_beds * 100.0 / NULLIF(hb.total_beds, 0) > 80 THEN 'High Occupancy'
    WHEN bs.occupied_beds * 100.0 / NULLIF(hb.total_beds, 0) BETWEEN 60 AND 80 THEN 'Moderate Occupancy'
    WHEN bs.occupied_beds * 100.0 / NULLIF(hb.total_beds, 0) < 60 THEN 'Low Occupancy'
    ELSE 'Unknown'
  END AS occupancy_category,
  
  CASE 
    WHEN fs.avg_patient_rating >= 4 THEN 'Excellent'
    WHEN fs.avg_patient_rating BETWEEN 3 AND 4 THEN 'Good'
    WHEN fs.avg_patient_rating < 3 THEN 'Needs Improvement'
    ELSE 'No Rating'
  END AS satisfaction_category,
  
  CURRENT_TIMESTAMP() AS created_at
  
FROM hospital_base hb
LEFT JOIN visit_stats vs ON hb.hospital_id = vs.hospital_id
LEFT JOIN bed_stats bs ON hb.hospital_id = bs.hospital_id
LEFT JOIN icu_latest icu ON hb.hospital_id = icu.hospital_id
LEFT JOIN department_stats dept ON hb.hospital_id = dept.hospital_id
LEFT JOIN feedback_stats fs ON hb.hospital_id = fs.hospital_id
LEFT JOIN referral_sent rs ON hb.hospital_id = rs.hospital_id
LEFT JOIN referral_received rr ON hb.hospital_id = rr.hospital_id
GROUP BY 
  hb.hospital_id, hb.hospital_name, hb.hospital_type, hb.governorate, hb.district, hb.manager_name,
  hb.total_beds, hb.icu_capacity, bs.total_beds_tracked, bs.available_beds, bs.occupied_beds,
  bs.icu_beds, bs.standard_beds, icu.latest_icu_occupied, icu.latest_icu_available, icu.latest_icu_update,
  dept.total_departments, dept.has_emergency_dept, vs.total_visits, vs.outpatient_visits, vs.inpatient_visits,
  vs.followup_visits, vs.unique_patients, vs.unique_doctors, vs.avg_waiting_time_minutes, vs.first_visit_date,
  vs.last_visit_date, vs.total_revenue, vs.avg_revenue_per_visit, fs.feedback_count, fs.avg_patient_rating,
  fs.min_rating, fs.max_rating, fs.positive_feedback_count, fs.negative_feedback_count,
  rs.referrals_sent, rr.referrals_received

In [0]:
# %sql
# CREATE OR REPLACE TABLE medical_insurance.gold.hospital_performance AS
# WITH hospital_base AS (
#   SELECT 
#     h.hospital_id,
#     h.hospital_name,
#     h.hospital_type,
#     h.governorate,
#     h.district,
#     h.total_beds,
#     h.icu_capacity,
#     h.manager_name
#   FROM medical_insurance.silver.hospital_silver h
# ),
# visit_stats AS (
#   SELECT 
#     v.hospital_id,
#     COUNT(DISTINCT v.visit_id) AS total_visits,
#     COUNT(DISTINCT CASE WHEN v.visit_type = 'Outpatient' THEN v.visit_id END) AS outpatient_visits,
#     COUNT(DISTINCT CASE WHEN v.visit_type = 'Inpatient' THEN v.visit_id END) AS inpatient_visits,
#     COUNT(DISTINCT CASE WHEN v.visit_type = 'Follow-up' THEN v.visit_id END) AS followup_visits,
#     COUNT(DISTINCT v.patient_id) AS unique_patients,
#     COUNT(DISTINCT v.doctor_id) AS unique_doctors,
#     AVG(v.waiting_time) AS avg_waiting_time_minutes,
#     SUM(v.total_amount) AS total_revenue,
#     AVG(v.total_amount) AS avg_revenue_per_visit,
#     MIN(v.visit_date) AS first_visit_date,
#     MAX(v.visit_date) AS last_visit_date
#   FROM medical_insurance.silver.visit_silver v
#   GROUP BY v.hospital_id
# ),
# bed_stats AS (
#   SELECT 
#     b.hospital_id,
#     COUNT(DISTINCT b.bed_id) AS total_beds_tracked,
#     COUNT(DISTINCT CASE WHEN b.availability_status = 'Available' THEN b.bed_id END) AS available_beds,
#     COUNT(DISTINCT CASE WHEN b.availability_status = 'Occupied' THEN b.bed_id END) AS occupied_beds,
#     COUNT(DISTINCT CASE WHEN b.bed_type = 'ICU' THEN b.bed_id END) AS icu_beds,
#     COUNT(DISTINCT CASE WHEN b.bed_type = 'Standard' THEN b.bed_id END) AS standard_beds
#   FROM medical_insurance.silver.bed_silver b
#   GROUP BY b.hospital_id
# ),
# icu_latest AS (
#   SELECT 
#     hospital_id,
#     occupied_beds AS latest_icu_occupied,
#     available_beds AS latest_icu_available,
#     update_time AS latest_icu_update
#   FROM (
#     SELECT 
#       hospital_id,
#       occupied_beds,
#       available_beds,
#       update_time,
#       ROW_NUMBER() OVER (PARTITION BY hospital_id ORDER BY update_time DESC) AS rn
#     FROM medical_insurance.silver.icu_status_silver
#   )
#   WHERE rn = 1
# ),
# department_stats AS (
#   SELECT 
#     dept.hospital_id,
#     COUNT(DISTINCT dept.department_id) AS total_departments,
#     COUNT(DISTINCT CASE WHEN dept.department_name = 'Emergency' THEN dept.department_id END) AS has_emergency_dept
#   FROM medical_insurance.silver.department_silver dept
#   GROUP BY dept.hospital_id
# ),
# feedback_stats AS (
#   SELECT 
#     f.hospital_id,
#     COUNT(DISTINCT f.feedback_id) AS feedback_count,
#     AVG(f.rating) AS avg_patient_rating,
#     MIN(f.rating) AS min_rating,
#     MAX(f.rating) AS max_rating,
#     COUNT(DISTINCT CASE WHEN f.rating >= 4 THEN f.feedback_id END) AS positive_feedback_count,
#     COUNT(DISTINCT CASE WHEN f.rating <= 2 THEN f.feedback_id END) AS negative_feedback_count
#   FROM medical_insurance.silver.patient_feedback_silver f
#   GROUP BY f.hospital_id
# ),
# referral_stats AS (
#   SELECT 
#     from_hospital_id AS hospital_id,
#     COUNT(DISTINCT referral_id) AS referrals_sent
#   FROM medical_insurance.silver.referral_silver
#   GROUP BY from_hospital_id
#   UNION ALL
#   SELECT 
#     to_hospital_id AS hospital_id,
#     COUNT(DISTINCT referral_id) AS referrals_received
#   FROM medical_insurance.silver.referral_silver
#   GROUP BY to_hospital_id
# )
# SELECT 
#   hb.hospital_id,
#   hb.hospital_name,
#   hb.hospital_type,
#   hb.governorate,
#   hb.district,
#   hb.manager_name,
  
#   -- Capacity metrics
#   hb.total_beds,
#   hb.icu_capacity,
#   COALESCE(bs.total_beds_tracked, 0) AS beds_in_system,
#   COALESCE(bs.available_beds, 0) AS current_available_beds,
#   COALESCE(bs.occupied_beds, 0) AS current_occupied_beds,
#   CASE 
#     WHEN hb.total_beds > 0 THEN ROUND((bs.occupied_beds * 100.0 / hb.total_beds), 2)
#     ELSE 0
#   END AS bed_occupancy_rate_pct,
#   COALESCE(bs.icu_beds, 0) AS icu_beds_tracked,
#   COALESCE(bs.standard_beds, 0) AS standard_beds_tracked,
  
#   -- ICU metrics
#   COALESCE(icu.latest_icu_occupied, 0) AS latest_icu_occupied,
#   COALESCE(icu.latest_icu_available, 0) AS latest_icu_available,
#   CASE 
#     WHEN hb.icu_capacity > 0 THEN ROUND((icu.latest_icu_occupied * 100.0 / hb.icu_capacity), 2)
#     ELSE 0
#   END AS icu_occupancy_rate_pct,
#   icu.latest_icu_update,
  
#   -- Department metrics
#   COALESCE(dept.total_departments, 0) AS total_departments,
#   CASE WHEN dept.has_emergency_dept > 0 THEN 'Yes' ELSE 'No' END AS has_emergency_department,
  
#   -- Visit metrics
#   COALESCE(vs.total_visits, 0) AS total_visits,
#   COALESCE(vs.outpatient_visits, 0) AS outpatient_visits,
#   COALESCE(vs.inpatient_visits, 0) AS inpatient_visits,
#   COALESCE(vs.followup_visits, 0) AS followup_visits,
#   COALESCE(vs.unique_patients, 0) AS unique_patients_served,
#   COALESCE(vs.unique_doctors, 0) AS unique_doctors_count,
#   ROUND(COALESCE(vs.avg_waiting_time_minutes, 0), 2) AS avg_waiting_time_minutes,
#   vs.first_visit_date,
#   vs.last_visit_date,
  
#   -- Revenue metrics
#   ROUND(COALESCE(vs.total_revenue, 0), 2) AS total_revenue,
#   ROUND(COALESCE(vs.avg_revenue_per_visit, 0), 2) AS avg_revenue_per_visit,
#   CASE 
#     WHEN vs.total_visits > 0 THEN ROUND((vs.total_revenue / vs.total_visits), 2)
#     ELSE 0
#   END AS revenue_per_visit,
  
#   -- Patient satisfaction metrics
#   COALESCE(fs.feedback_count, 0) AS feedback_count,
#   ROUND(COALESCE(fs.avg_patient_rating, 0), 2) AS avg_patient_rating,
#   fs.min_rating,
#   fs.max_rating,
#   COALESCE(fs.positive_feedback_count, 0) AS positive_feedback_count,
#   COALESCE(fs.negative_feedback_count, 0) AS negative_feedback_count,
#   CASE 
#     WHEN fs.feedback_count > 0 THEN ROUND((fs.positive_feedback_count * 100.0 / fs.feedback_count), 2)
#     ELSE 0
#   END AS positive_feedback_rate_pct,
  
#   -- Referral metrics
#   SUM(CASE WHEN ref.referrals_sent IS NOT NULL THEN ref.referrals_sent ELSE 0 END) AS referrals_sent,
#   SUM(CASE WHEN ref.referrals_received IS NOT NULL THEN ref.referrals_received ELSE 0 END) AS referrals_received,
  
#   -- Performance indicators
#   CASE 
#     WHEN vs.total_visits > 10000 THEN 'High Volume'
#     WHEN vs.total_visits BETWEEN 5000 AND 10000 THEN 'Medium Volume'
#     WHEN vs.total_visits < 5000 THEN 'Low Volume'
#     ELSE 'No Activity'
#   END AS volume_category,
  
#   CASE 
#     WHEN bs.occupied_beds * 100.0 / NULLIF(hb.total_beds, 0) > 80 THEN 'High Occupancy'
#     WHEN bs.occupied_beds * 100.0 / NULLIF(hb.total_beds, 0) BETWEEN 60 AND 80 THEN 'Moderate Occupancy'
#     WHEN bs.occupied_beds * 100.0 / NULLIF(hb.total_beds, 0) < 60 THEN 'Low Occupancy'
#     ELSE 'Unknown'
#   END AS occupancy_category,
  
#   CASE 
#     WHEN fs.avg_patient_rating >= 4 THEN 'Excellent'
#     WHEN fs.avg_patient_rating BETWEEN 3 AND 4 THEN 'Good'
#     WHEN fs.avg_patient_rating < 3 THEN 'Needs Improvement'
#     ELSE 'No Rating'
#   END AS satisfaction_category,
  
#   CURRENT_TIMESTAMP() AS created_at
  
# FROM hospital_base hb
# LEFT JOIN visit_stats vs ON hb.hospital_id = vs.hospital_id
# LEFT JOIN bed_stats bs ON hb.hospital_id = bs.hospital_id
# LEFT JOIN icu_latest icu ON hb.hospital_id = icu.hospital_id
# LEFT JOIN department_stats dept ON hb.hospital_id = dept.hospital_id
# LEFT JOIN feedback_stats fs ON hb.hospital_id = fs.hospital_id
# LEFT JOIN referral_stats ref ON hb.hospital_id = ref.hospital_id
# GROUP BY 
#   hb.hospital_id, hb.hospital_name, hb.hospital_type, hb.governorate, hb.district, hb.manager_name,
#   hb.total_beds, hb.icu_capacity, bs.total_beds_tracked, bs.available_beds, bs.occupied_beds,
#   bs.icu_beds, bs.standard_beds, icu.latest_icu_occupied, icu.latest_icu_available, icu.latest_icu_update,
#   dept.total_departments, dept.has_emergency_dept, vs.total_visits, vs.outpatient_visits, vs.inpatient_visits,
#   vs.followup_visits, vs.unique_patients, vs.unique_doctors, vs.avg_waiting_time_minutes, vs.first_visit_date,
#   vs.last_visit_date, vs.total_revenue, vs.avg_revenue_per_visit, fs.feedback_count, fs.avg_patient_rating,
#   fs.min_rating, fs.max_rating, fs.positive_feedback_count, fs.negative_feedback_count

In [0]:
%sql
-- Display sample records from hospital performance gold table
SELECT 
  hospital_name,
  hospital_type,
  total_visits,
  bed_occupancy_rate_pct,
  icu_occupancy_rate_pct,
  avg_patient_rating,
  total_revenue,
  volume_category,
  satisfaction_category
FROM medical_insurance.gold.hospital_performance
ORDER BY total_visits DESC
LIMIT 10

In [0]:
%sql
-- Summary statistics by hospital type
SELECT 
  hospital_type,
  COUNT(DISTINCT hospital_id) AS total_hospitals,
  ROUND(AVG(total_visits), 2) AS avg_visits_per_hospital,
  ROUND(AVG(bed_occupancy_rate_pct), 2) AS avg_bed_occupancy_pct,
  ROUND(AVG(icu_occupancy_rate_pct), 2) AS avg_icu_occupancy_pct,
  ROUND(AVG(avg_patient_rating), 2) AS avg_rating,
  ROUND(AVG(total_revenue), 2) AS avg_revenue_per_hospital,
  ROUND(AVG(avg_waiting_time_minutes), 2) AS avg_wait_time,
  volume_category,
  satisfaction_category
FROM medical_insurance.gold.hospital_performance
GROUP BY hospital_type, volume_category, satisfaction_category
ORDER BY hospital_type, volume_category

In [0]:
%sql
select * from medical_insurance.gold.hospital_performance